In [19]:
MAX_LEN      = 90
MAX_VOCAB    = 8000
BATCH        = 64
EPOCHS       = 30
PATIENCE     = 5
LR           = 2e-3
WEIGHT_DECAY = 1e-2
ALPHA_FOCAL  = 0.75
GAMMA_FOCAL  = 2
SMOOTH       = 0.05
HARD_EPOCH   = 1
SEED         = 42
MODEL_NAME   = 'best_xss_focal.pth'

import torch, random, numpy as np, pandas as pd, re, html, urllib.parse
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             average_precision_score)
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [20]:
def xss_preprocess(text: str) -> str:
    if pd.isna(text): return ''
    s = str(text).strip()
    s = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]+', ' ', s)
    s = urllib.parse.unquote(s)
    s = html.unescape(s)
    s = re.sub(r'\\u([0-9a-fA-F]{4})', lambda m: chr(int(m.group(1), 16)), s)
    s = s.replace('<', ' < ').replace('>', ' > ')
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s

def xss_tokenize(text: str):
    return re.findall(r"\w+|</?\w+|[<>\"'=;:/.\-]|\S", text)[:MAX_LEN]

In [21]:
df = pd.read_csv('data/XSS_dataset.csv')
texts, labels = df['sentence'].astype(str).values, df['Label'].astype(int).values

train_txt, tmp_txt, y_train, y_tmp = train_test_split(
        texts, labels, test_size=0.4, stratify=labels, random_state=SEED)
val_txt, test_txt, y_val, y_test   = train_test_split(
        tmp_txt, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED)

In [22]:
train_tok = [xss_tokenize(xss_preprocess(t)) for t in train_txt]
val_tok   = [xss_tokenize(xss_preprocess(t)) for t in val_txt]
# test_tok  = [xss_tokenize(xss_preprocess(t)) for t in test_txt]

counter = Counter(tok for sent in train_tok for tok in sent)
vocab = ['<PAD>', '<UNK>'] + [w for w, _ in counter.most_common(MAX_VOCAB-2)]
word2idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
print(vocab_size)

6175


In [23]:
def encode(tokens_list):
    seqs = []
    for toks in tokens_list:
        seq = [word2idx.get(t, 1) for t in toks][:MAX_LEN]
        seq += [0] * (MAX_LEN - len(seq))
        seqs.append(seq)
    return torch.LongTensor(seqs)

X_train, X_val = encode(train_tok), encode(val_tok)
y_train, y_val, y_test = torch.LongTensor(y_train), torch.LongTensor(y_val), torch.LongTensor(y_test)

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_train, y_train), batch_size=BATCH, shuffle=True, drop_last=True)
val_loader   = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_val, y_val), batch_size=BATCH*2)
# test_loader  = torch.utils.data.DataLoader(
#     torch.utils.data.TensorDataset(X_test, y_test), batch_size=BATCH*2)

In [24]:
import torch.nn as nn
import torch.nn.functional as F

class XSSDetector(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb   = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.drop1 = nn.Dropout(0.2)
        # self.convs = nn.ModuleList([nn.Conv1d(128, 128, k, padding=k//2) for k in [3,4,5]])
        self.convs = nn.ModuleList([nn.Conv1d(128, 128, k, padding=(k//2))   for k in [3,5,7] ])
        self.attn  = nn.Sequential(nn.Linear(128*3, 64), nn.Tanh(), nn.Linear(64, 1))
        self.lstm  = nn.LSTM(128*3, 256, batch_first=True, bidirectional=True)
        self.fc    = nn.Sequential(
                        nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.4),
                        nn.Linear(128, 2))
    def forward(self, x):
        x = self.emb(x)
        x = self.drop1(x).permute(0,2,1)
        h = torch.cat([F.relu(c(x)) for c in self.convs], dim=1)
        h = h.permute(0,2,1)
        a = self.attn(h)
        a = torch.softmax(a, dim=1)
        h = h * a
        h,_ = self.lstm(h)
        h = torch.max(h, dim=1)[0]
        return self.fc(h)

net = XSSDetector(vocab_size).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in net.parameters()):,}')

Parameters: 2,441,987


In [25]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=ALPHA_FOCAL, gamma=GAMMA_FOCAL, smooth=SMOOTH):
        super().__init__()
        self.a, self.g, self.s = alpha, gamma, smooth
    def forward(self, logits, y):
        y_smooth = torch.zeros_like(logits).scatter_(1, y.view(-1,1), 1)
        y_smooth = y_smooth * (1-self.s) + self.s/2.
        log_p = F.log_softmax(logits, dim=1)
        p = torch.exp(log_p)
        loss = -y_smooth * torch.pow(1-p, self.g) * log_p
        alpha_t = torch.tensor([self.a, 1-self.a]).to(logits.device).view(1,2)
        loss = alpha_t * loss
        return loss.sum(dim=1).mean()

criterion = FocalLoss()
optimizer = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [26]:
best_auc, patience = 0, 0
for epoch in range(1, EPOCHS+1):
    net.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(net(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
        optimizer.step()
    scheduler.step()


    net.eval()
    with torch.no_grad():
        probs = torch.softmax(net(X_val.to(DEVICE)), dim=1)[:,1].cpu().numpy()
    auc = roc_auc_score(y_val.numpy(), probs)
    print(f'Ep {epoch:02d}  Val-AUC={auc:.4f}')
    if auc > best_auc:
        best_auc, patience = auc, 0
        # torch.save(net.state_dict(), MODEL_NAME)
        torch.save({'model': net.state_dict(),
            'word2idx': word2idx,
            'max_len': MAX_LEN},
           'best_xss_focal_FULL.pth')
    else:
        patience += 1
        if patience >= PATIENCE:
            print('Early-stop')
            break

Ep 01  Val-AUC=0.9998
Ep 02  Val-AUC=0.9999
Ep 03  Val-AUC=0.9999
Ep 04  Val-AUC=0.9998
Ep 05  Val-AUC=1.0000
Ep 06  Val-AUC=1.0000
Ep 07  Val-AUC=0.9999
Ep 08  Val-AUC=1.0000
Ep 09  Val-AUC=0.9999
Ep 10  Val-AUC=1.0000
Ep 11  Val-AUC=1.0000
Ep 12  Val-AUC=1.0000
Ep 13  Val-AUC=1.0000
Ep 14  Val-AUC=1.0000
Ep 15  Val-AUC=1.0000
Ep 16  Val-AUC=1.0000
Ep 17  Val-AUC=0.9998
Early-stop


In [27]:
net.load_state_dict(torch.load(MODEL_NAME))
net.eval()
with torch.no_grad():
    probs_train = torch.softmax(net(X_train.to(DEVICE)), dim=1)[:,1].cpu()
margin = torch.abs(probs_train - 0.5)
hard_idx = torch.where(margin < 0.3)[0]
if len(hard_idx) > 0:
    hard_set = torch.utils.data.TensorDataset(X_train[hard_idx], y_train[hard_idx])
    hard_loader = torch.utils.data.DataLoader(hard_set, batch_size=BATCH, shuffle=True)
    optimizer = torch.optim.AdamW(net.parameters(), lr=LR/5)
    for _ in range(HARD_EPOCH):
        for x, y in hard_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(net(x), y)
            loss.backward()
            optimizer.step()

In [28]:
test_tok  = [xss_tokenize(xss_preprocess(t)) for t in test_txt]
X_test = encode(test_tok)
test_loader  = torch.utils.data.DataLoader(
     torch.utils.data.TensorDataset(X_test, y_test), batch_size=BATCH*2)
net.load_state_dict(torch.load(MODEL_NAME))
net.eval()
with torch.no_grad():
    test_prob = torch.softmax(net(X_test.to(DEVICE)), dim=1)[:,1].cpu().numpy()
y_true = y_test.numpy()
y_pred = (test_prob > 0.5).astype(int)

print('========== Final Test Metrics ==========')
print(f'AUC               : {roc_auc_score(y_true, test_prob):.4f}')
print(f'Accuracy          : {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision (macro) : {precision_score(y_true, y_pred, average="macro"):.4f}')
print(f'Recall    (macro) : {recall_score(y_true, y_pred, average="macro"):.4f}')
print(f'F1        (macro) : {f1_score(y_true, y_pred, average="macro"):.4f}')
print(f'PR-AUC (class-1)  : {average_precision_score(y_true, test_prob):.4f}')
print('---------- per-class ----------')
from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=[0,1])
df_cls = pd.DataFrame({'Class':[0,1], 'Precision':prec, 'Recall':rec, 'F1':f1, 'Support':sup})
print(df_cls.to_string(index=False))
print('---------- confusion ----------')
print(confusion_matrix(y_true, y_pred))

========== Final Test Metrics ==========
AUC               : 0.9999
Accuracy          : 0.9982
Precision (macro) : 0.9981
Recall    (macro) : 0.9982
F1        (macro) : 0.9982
PR-AUC (class-1)  : 0.9999
---------- per-class ----------
 Class  Precision   Recall       F1  Support
     0   0.996840 0.999208 0.998023     1263
     1   0.999321 0.997288 0.998303     1475
---------- confusion ----------
[[1262    1]
 [   4 1471]]


In [35]:
import torch, pandas as pd, numpy as np, re, html, urllib.parse
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix,
                             average_precision_score)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ==================================================
# 1- مسیر فایل‌ها و ستون‌ها
# ==================================================
CSV_PATH    = 'data/xss_da.csv'  # <==== عوض کنید
COL_PAYLOAD = 'sentence'                 # <==== نام ستون متن
COL_LABEL   = 'Label'                   # <==== نام ستون برچسب
MODEL_PATH  = 'best_xss_focal.pth'      # <==== چک‌پوینت مدل + word2idx
MAX_LEN     = 90                        # همان مقدار training
BATCH_INF   = 128                      # کم کن اگر OOM گرفتی

In [36]:
# ==================================================
# 2- Load CSV + drop NaN
# ==================================================
df_new = pd.read_csv(CSV_PATH).dropna(subset=[COL_PAYLOAD, COL_LABEL])
payloads = df_new[COL_PAYLOAD].astype(str).values
labels   = df_new[COL_LABEL].astype(int).values
print(f'New dataset: {len(payloads)} samples')

New dataset: 41746 samples


In [38]:
# ==================================================
# 3- Load checkpoint (model + word2idx)
# ==================================================
ckpt = torch.load('best_xss_focal_FULL.pth', map_location=DEVICE)
word2idx = ckpt['word2idx']
vocab_size = len(word2idx)
print('Vocab loaded, size:', vocab_size)

Vocab loaded, size: 6175


In [39]:
# ==================================================
# 5- Tokenize + Encode with EXACT same vocab
# ==================================================
tok_new = [xss_tokenize(xss_preprocess(p)) for p in payloads]

def encode_with_vocab(tokens_list, w2i):
    seqs = []
    for toks in tokens_list:
        seq = [w2i.get(t, 1) for t in toks][:MAX_LEN]   # 1 = <UNK>
        seq += [0] * (MAX_LEN - len(seq))               # 0 = <PAD>
        seqs.append(seq)
    return torch.LongTensor(seqs)

X_new = encode_with_vocab(tok_new, word2idx)
y_new = torch.LongTensor(labels)
print('Encoding finished, shape:', X_new.shape)

Encoding finished, shape: torch.Size([41746, 90])


In [40]:
# ==================================================
# 6- Build DataLoader (batch-wise to avoid OOM)
# ==================================================
from torch.utils.data import TensorDataset, DataLoader

test_loader = DataLoader(
    TensorDataset(X_new, y_new),
    batch_size=BATCH_INF,
    shuffle=False,
    pin_memory=True
)

In [41]:


model = XSSDetector(vocab_size).to(DEVICE)
model.load_state_dict(ckpt['model'])   # وزن‌های شبکه
model.eval()

probs_list, labs_list = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        pb = torch.softmax(model(xb), dim=1)[:, 1].cpu()
        probs_list.append(pb)
        labs_list.append(yb)

probs = torch.cat(probs_list).numpy()
y_true = torch.cat(labs_list).numpy()

C:\Users\BARCODE\Desktop\JupyterProject\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [42]:
# ==================================================
# 8- Compute ALL metrics
# ==================================================
y_pred = (probs > 0.5).astype(int)

print('========== Evaluation on NEW dataset ==========')
print(f'AUC               : {roc_auc_score(y_true, probs):.4f}')
print(f'Accuracy          : {accuracy_score(y_true, y_pred):.4f}')
print(f'Precision (macro) : {precision_score(y_true, y_pred, average="macro"):.4f}')
print(f'Recall    (macro) : {recall_score(y_true, y_pred, average="macro"):.4f}')
print(f'F1        (macro) : {f1_score(y_true, y_pred, average="macro"):.4f}')
print(f'PR-AUC (attack)   : {average_precision_score(y_true, probs):.4f}')

prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=[0,1])
df_cls = pd.DataFrame({'Class': [0, 1], 'Precision': prec, 'Recall': rec, 'F1': f1, 'Support': sup})
print(df_cls.to_string(index=False))

print('Confusion matrix (true→rows, pred→cols):')
print(confusion_matrix(y_true, y_pred))

========== Evaluation on NEW dataset ==========
AUC               : 0.9995
Accuracy          : 0.9949
Precision (macro) : 0.9863
Recall    (macro) : 0.9965
F1        (macro) : 0.9913
PR-AUC (attack)   : 0.9928
 Class  Precision   Recall       F1  Support
     0   0.999766 0.994036 0.996893    34373
     1   0.972919 0.998915 0.985746     7373
Confusion matrix (true→rows, pred→cols):
[[34168   205]
 [    8  7365]]
